# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This helps identify the structure and contents of the dataset prior to extraction.

The `dataset.metadata` object provides access to all record sets via their `@id`. Below, we'll enumerate each record set, its name, and its fields and columns (by their `@id`).

In [ ]:
# List all record sets and their fields and columns by @id
from pprint import pprint

if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = []

if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '')}")
        fields = rs.get('fields', [])
        if fields:
            print("  Fields:")
            for field in fields:
                print(f"    - {field['@id']} ({field.get('name', '')})")
        columns = rs.get('columns', [])
        if columns:
            print("  Columns:")
            for column in columns:
                print(f"    - {column['@id']} ({column.get('name', '')})")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

First, let's list the available record set `@id`s.

In [ ]:
# Find all available record set IDs
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = []

record_set_ids = [rs['@id'] for rs in record_sets]
print("Available record_set @id's:")
pprint(record_set_ids)

In [ ]:
# Load records from all record sets into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  No records found for record set {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample:\n{df.head(2) if not df.empty else df}")

# For demonstration, pick the first record set if available
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nMain record set selected for analysis: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

Below, we demonstrate these analyses on a numeric field in the main record set. Remember, refer to all fields by their `@id` as shown above.

In [ ]:
# For demonstration, we try to select a numeric field from the main DataFrame
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try to infer a numeric field
    numeric_field_candidates = df.select_dtypes(include='number').columns.tolist()
    print(f"Numeric field candidates for EDA: {numeric_field_candidates}")
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]

        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to use a group field, e.g. the first non-numeric column
        non_numeric_fields = [c for c in df.columns if c not in numeric_field_candidates]
        group_field = non_numeric_fields[0] if non_numeric_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA in the main record set.")
else:
    print("No record sets with data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we show a histogram of the chosen numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=40, kde=True, color='royalblue')
    plt.title(f'Distribution of {numeric_field} in {main_record_set_id}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
else:
    print("No numeric field to display histogram.")

## 6. Conclusion
In this notebook, we loaded metadata and tabular data via the FAIR\(^2\) Croissant schema, reviewed the data structure, extracted records using their `@id`, and performed exploratory analysis and visualization. Use the field and column `@id`s in this workflow for robust downstream processing and reproducible scientific workflows.